## MC vs Split CP 시뮬레이션 - 고정 grid 입력별 conditional coverage

설계 기준 문서: 프로젝트 루트의 `시뮬.md`

### 연구 목적

Random Forest(RF)를 기준 모델로 하여 두 예측구간 방법을 비교

- **MC** : 기준 RF $\widehat f$와 CV 잔차분산 $\widehat\sigma^2$로 가상 training 반응값 $Y^{*}$를 만들고, RF를 $B$번 재학습한 예측값에 새 Gaussian 오차를 더한 표본의 분위수로 구간 생성
- **Split CP** : 매 반복 새 calibration 데이터의 절대잔차 score에서 순위 분위수 $\widehat q_\alpha$를 구해 $\widehat f(x)\pm\widehat q_\alpha$ 구간 생성

### 평가 대상

고정 grid 입력 $\underline x_i$ 25개(두 축 $\{-1,-0.5,0,0.5,1\}$의 모든 조합)에서, 매 반복 새로 생성한 반응값 $Y_i^{grid,(r)}$에 대한

$$
\widehat c_{h,\alpha}(\underline x_i)=\frac1R\sum_{r=1}^R \mathbf 1\{L_{h,i,r,\alpha}\le Y_i^{grid,(r)}\le U_{h,i,r,\alpha}\},
\qquad
\overline\ell_{h,\alpha}(\underline x_i)=\frac1R\sum_{r=1}^R (U_{h,i,r,\alpha}-L_{h,i,r,\alpha})
$$

($h\in\{MC,CP\}$, $\alpha\in\{0.10,0.05\}$)
- Training을 고정한 상태의 X-conditional coverage이며, training 자체의 변동은 평균 X
- Split CP가 모든 $x$에서 목표 conditional coverage를 보장하는 것은 아님

### 반복 구조

```text
for scenario:                       # 4개 DGP
    training 생성 → 기준 RF 학습 → CV 잔차로 sigma2_hat 추정 → grid 예측 준비
    for r in 1..R:
        grid 반응값 25개, calibration 1000개를 true DGP에서 새로 생성
        for b in 1..B:
            고정 training X에서 가상 Y* 생성 → 새 RF 학습 → grid 25개 예측 + 새 Gaussian 오차
        같은 B×25 MC 표본에서 두 alpha의 MC 구간
        기준 RF + 이번 calibration에서 두 alpha의 CP 구간
        같은 grid 반응값으로 두 방법·두 alpha의 포함 여부와 길이 기록
```

| 고정 | 매 반복 재생성 |
| --- | --- |
| 원래 training, 기준 RF $\widehat f$, $\widehat\sigma^2$, grid 입력 | calibration X·Y, grid 반응값, MC 가상 Y*, MC 재학습 RF, MC 새 관측오차 |


## 1. Imports, 실행 환경, 프로젝트 루트

노트북은 `notebooks/`에 있지만 결과는 프로젝트 루트의 `results/`에 저장한다. 작업 디렉터리와 무관하게 현재 디렉터리 또는 상위 디렉터리 중 `시뮬.md`가 있는 곳을 `PROJECT_ROOT`로 사용한다.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

# 현재 위치에서 위로 올라가며 시뮬.md가 있는 폴더를 찾는다 → 프로젝트 루트
PROJECT_ROOT = None
here = Path.cwd().resolve()
for folder in [here] + list(here.parents):
    if (folder / '시뮬.md').exists():
        PROJECT_ROOT = folder
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('시뮬.md를 찾지 못했습니다. 프로젝트 폴더 안에서 노트북을 실행하세요.')

print('PROJECT_ROOT :', PROJECT_ROOT)
print('python       :', sys.version.split()[0])
print('numpy        :', np.__version__)
print('pandas       :', pd.__version__)
print('scikit-learn :', sklearn.__version__)

PROJECT_ROOT : C:\Users\cho03\Desktop\조민석\연구
python       : 3.13.15
numpy        : 2.5.3
pandas       : 3.0.6
scikit-learn : 1.9.1


## 2. 설정

### 확정 설계값 (`시뮬.md`)

연구 설계에서 확정된 값. 바꾸면 다른 실험이 된다.

### 구현 기본값 (변경 가능한 제안값)

`시뮬.md`에 수치가 없어 실행 가능한 초안을 위해 정한 값. 연구에서 확정된 값이 아니다.

- 튜닝 후보: `GRID_MIN_SAMPLES_LEAF` × `GRID_MAX_FEATURES` × `GRID_MAX_SAMPLES` = 24개 조합

### 튜닝·분산 추정 방식 (확정)

| 단계 | 방식 |
| --- | --- |
| 기준 RF 튜닝 | OOB: 24개 조합을 모두 학습하고 OOB MSE가 가장 작은 RF를 그대로 최종 모델로 사용 |
| $\widehat\sigma^2$ 추정 | 10-fold CV, 기준 RF의 조합을 고정 (fold 안에서 다시 튜닝하지 않음) |
| MC 재학습 튜닝 | OOB: 가상 데이터마다 24개 조합을 다시 비교 (기준 RF와 같은 절차) |

### 실행 모드

- `'smoke'` : 구조 검증용 작은 실행 (R=2, B=5, 트리 30개). **연구 결과가 아니다.** 결과는 `results/check/<scenario>/` (실제 smoke 실행은 `simulation.ipynb` → `results/smoke/`)
- `'full'` : 설계값 그대로 (R=200, B=200, 트리 1000개). 결과는 `results/<scenario>/`

`RUN_SCENARIOS`로 이 컴퓨터에서 돌릴 scenario를 고른다 (scenario별로 다른 컴퓨터에서 실행).

In [2]:
# ----- 확정 설계값 (시뮬.md) -----
SCENARIOS = [
    'linear_homo_gaussian',
    'linear_homo_student_t',
    'nonlinear_homo_gaussian',
    'nonlinear_homo_student_t',
]
N_TRAIN = 1000
N_CAL = 1000
ALPHAS = [0.10, 0.05]

# ----- 구현 기본값 (변경 가능) -----
MASTER_SEED = 0
MC_QUANTILE_METHOD = 'linear'
CV_FOLDS = 10

# 튜닝하지 않는 RF 설정 (트리 수는 아래 실행 모드에서 정함)
MAX_DEPTH = None

# ----- 실행 모드 -----
RUN_MODE = 'smoke'   # 'smoke' 또는 'full'

if RUN_MODE == 'smoke':
    R = 2
    B = 5
    N_TREES = 30     # RF 하나 안에 들어가는 결정트리의 개수
    RESULTS_DIR = PROJECT_ROOT / 'results' / 'check'   # 확인용 노트북 결과 (simulation.ipynb의 results/smoke와 분리)
elif RUN_MODE == 'full':
    R = 200
    B = 200
    N_TREES = 1000
    RESULTS_DIR = PROJECT_ROOT / 'results'
else:
    raise ValueError("RUN_MODE는 'smoke' 또는 'full'")

# 이 컴퓨터에서 실행할 scenario 목록
# smoke : 4개 모두 실행
# full  : 컴퓨터마다 하나씩 골라 적는다. 예) RUN_SCENARIOS = ['linear_homo_gaussian']
RUN_SCENARIOS = [
    'linear_homo_gaussian',
    'linear_homo_student_t',
    'nonlinear_homo_gaussian',
    'nonlinear_homo_student_t',
]

print('RUN_MODE      :', RUN_MODE)
print('R, B, 트리 수 :', R, B, N_TREES)
print('RUN_SCENARIOS :', RUN_SCENARIOS)
print('RESULTS_DIR   :', RESULTS_DIR)

RUN_MODE      : smoke
R, B, 트리 수 : 2 5 30
RUN_SCENARIOS : ['linear_homo_gaussian', 'linear_homo_student_t', 'nonlinear_homo_gaussian', 'nonlinear_homo_student_t']
RESULTS_DIR   : C:\Users\cho03\Desktop\조민석\연구\results\check


## 3. 난수 관리 — master seed와 단계·반복 ID 조합

모든 난수 흐름은 키 `[MASTER_SEED, scenario_id, stage_id, r, b, fold]`로 구분한다.

- `scenario_id`, `stage_id`는 아래 고정 매핑을 사용한다. 실행 목록을 바꿔도 ID는 바뀌지 않는다.
- `r=1..R`, `b=1..B`, `fold=1..CV_FOLDS`이고, 해당 단계에 필요 없는 인덱스는 0이다.
- 생성기는 `make_random_state` 하나로 통일한다: `RandomState(SeedSequence(key).generate_state(4))`
  - 데이터·오차 생성(`uniform`, `normal`, `standard_t`)과 sklearn `random_state`(RF, KFold)에 모두 쓴다.
  - `RandomState`는 NumPy 버전이 달라도 같은 난수를 내도록 보장되어, 여러 컴퓨터에서 나눠 실행할 때 안전하다.
  - 부를 때마다 같은 키에서 새 객체를 만든다 (한 객체를 여러 곳에서 이어 쓰지 않음).

| 단계 | stage_id | 인덱스 | 용도 |
| --- | ---: | --- | --- |
| `common_dataset` | 0 | 없음 | 원래 training X와 오차 |
| `common_training` | 1 | 없음 | 기준 RF 학습 |
| `cv_split` | 2 | 없음 | CV fold 분할 |
| `cv_model` | 3 | fold | fold별 RF 학습 |
| `cp_calibration` | 4 | r | calibration X·Y 생성 |
| `mc_dataset` | 5 | r, b | 가상 training 오차 |
| `mc_training` | 6 | r, b | MC RF 재학습 |
| `mc_new_error` | 7 | r, b | MC grid 예측에 더할 Gaussian 오차 |
| `test_error` | 8 | r | 실제 평가용 grid 오차 |

키 방식은 난수 흐름을 재현 가능하게 분리하기 위한 것이며, 유한한 의사난수로 수학적 독립성을 증명하는 것은 아니다.

In [3]:
SCENARIO_IDS = {
    'linear_homo_gaussian': 0,
    'linear_homo_student_t': 1,
    'nonlinear_homo_gaussian': 2,
    'nonlinear_homo_student_t': 3,
}
STAGE_IDS = {
    'common_dataset': 0,
    'common_training': 1,
    'cv_split': 2,
    'cv_model': 3,
    'cp_calibration': 4,
    'mc_dataset': 5,
    'mc_training': 6,
    'mc_new_error': 7,
    'test_error': 8,
}


def make_random_state(scenario, stage, r=0, b=0, fold=0): # 모든 난수에 쓰는 생성기
    # 키 = [MASTER_SEED, scenario_id, stage_id, r, b, fold] → 시드 재료(SeedSequence) → 정수 4개 → RandomState
    # 데이터·오차 생성과 sklearn(RF, KFold)의 random_state에 모두 쓴다. 부를 때마다 새 객체를 만든다.
    key = [MASTER_SEED, SCENARIO_IDS[scenario], STAGE_IDS[stage], r, b, fold]
    words = np.random.SeedSequence(key).generate_state(4)
    return np.random.RandomState(words)

# eg) eg =  make_random_state('linear_homo_gaussian', 'mc_dataset', r=1, b=1).normal(size=3)

## 4. DGP — 평균함수와 오차

$$
Y=m(\underline X)+\varepsilon,\qquad X_1,X_2\overset{iid}{\sim}U(-1,1)
$$

| 평균함수 | 코드 |
| --- | --- |
| $m_L(\underline x)=x_1+x_2$ | `X[:, 0] + X[:, 1]` |
| $m_N(\underline x)=\sqrt{2/3}\sin(\pi x_1)+x_2$ | `np.sqrt(2/3) * np.sin(np.pi * X[:, 0]) + X[:, 1]` |

<br>

| 오차 | 코드 |
| --- | --- |
| $\varepsilon_G\sim N(0,1)$ | `rs.normal(size=size)` |
| $\varepsilon_T=T/\sqrt3,\ T\sim t_3$ | `rs.standard_t(3, size=size) / np.sqrt(3)` |

두 오차 모두 평균 0, 분산 1 (표준편차 $\sigma_0=1$ 고정).

`X`는 `(n, 2)` 배열: `X[:, 0]`이 첫 번째 설명변수 $x_1$, `X[:, 1]`이 두 번째 설명변수 $x_2$, `X[i, :]`가 입력 벡터 하나.

In [4]:
def mean_function(scenario, X):
    # 참평균 m(x). scenario 이름이 linear로 시작하면 m_L, nonlinear로 시작하면 m_N
    if scenario.startswith('linear'):
        return X[:, 0] + X[:, 1]
    elif scenario.startswith('nonlinear'):
        return np.sqrt(2 / 3) * np.sin(np.pi * X[:, 0]) + X[:, 1]
    else:
        raise ValueError(scenario)


def draw_error(scenario, random_seed, size):
    # 참오차 ε (평균 0, 분산 1). scenario 이름 끝이 gaussian이면 N(0,1), student_t이면 t_3/sqrt(3)
    if scenario.endswith('gaussian'):
        return random_seed.normal(size=size)
    elif scenario.endswith('student_t'):
        return random_seed.standard_t(3, size=size) / np.sqrt(3)
    else:
        raise ValueError(scenario)


# 손으로 계산할 수 있는 점 몇 개로 확인
X_check = np.array([
    [0.0, 0.0],
    [0.5, 0.0],
    [1.0, 1.0],
    [-0.5, 0.5],
])
print('x            :', X_check.tolist())
print('linear 평균   :', mean_function('linear_homo_gaussian', X_check))
print('nonlinear 평균:', mean_function('nonlinear_homo_gaussian', X_check))
print('sqrt(2/3)    :', np.sqrt(2 / 3), '  ← nonlinear (0.5, 0)과 같아야 함')

x            : [[0.0, 0.0], [0.5, 0.0], [1.0, 1.0], [-0.5, 0.5]]
linear 평균   : [0.  0.5 2.  0. ]
nonlinear 평균: [ 0.          0.81649658  1.         -0.31649658]
sqrt(2/3)    : 0.816496580927726   ← nonlinear (0.5, 0)과 같아야 함


## 5. Grid 25개

두 축 $\{-1,-0.5,0,0.5,1\}$의 모든 조합. 바깥 루프가 $x_1$, 안쪽 루프가 $x_2$이고 `grid_id`는 0~24로 이 순서대로 고정한다.

In [5]:
GRID_AXIS = [-1.0, -0.5, 0.0, 0.5, 1.0]

grid_points = []
for x1 in GRID_AXIS:
    for x2 in GRID_AXIS:
        grid_points.append([x1, x2])

X_grid = np.array(grid_points)
N_GRID = len(X_grid)

grid_df = pd.DataFrame({
    'grid_id': np.arange(N_GRID),
    'x1': X_grid[:, 0],
    'x2': X_grid[:, 1],
})

print('X_grid.shape :', X_grid.shape)
grid_df

X_grid.shape : (25, 2)


,grid_id,x1,x2
0,0,-1.0,-1.0
1,1,-1.0,-0.5
2,2,-1.0,0.0
3,3,-1.0,0.5
4,4,-1.0,1.0
5,5,-0.5,-1.0
6,6,-0.5,-0.5
7,7,-0.5,0.0
8,8,-0.5,0.5
9,9,-0.5,1.0


## 6. Training 데이터 생성 — 첫 scenario로 따라가기

`linear_homo_gaussian` 하나로 먼저 따라간다. 생성기 `make_random_state(s, 'common_dataset')` **하나**에서 X를 먼저 뽑고, 같은 생성기에서 이어서 오차를 뽑는다.

$$
X_{j1},X_{j2}\sim U(-1,1),\qquad Y_j=m(\underline X_j)+\varepsilon_j,\qquad j=1,\ldots,1000
$$

In [6]:
random_seed = make_random_state('linear_homo_gaussian', 'common_dataset')
X_train = random_seed.uniform(-1, 1, size=(1000, 2))
eps_train = draw_error('linear_homo_gaussian', random_seed, 1000)
Y_train = mean_function('linear_homo_gaussian', X_train) + eps_train

pd.DataFrame({
    'x1': X_train[:, 0],
    'x2': X_train[:, 1],
    'y': Y_train[:],
})

,x1,x2,y
0,0.704293,0.557416,0.785362
1,-0.378332,-0.471905,-0.094927
2,0.098369,0.711712,0.173086
3,0.764764,0.576740,0.517632
4,-0.073880,-0.037704,0.528062
...,...,...,...
995,-0.992051,0.197562,-0.320407
996,-0.828187,0.348705,-0.230367
997,-0.910988,0.050766,-2.088646
998,0.173330,-0.007109,-0.043355


## 7. 기준 RF — OOB grid search

RF의 Fit은 튜닝을 포함한다:

1. 24개 조합 각각으로 RF를 전체 데이터에 학습 (`oob_score=True`)
2. 각 RF의 OOB 예측 `oob_prediction_`으로 OOB MSE $=\frac1n\sum_j (Y_j-\widehat f^{OOB}(X_j))^2$ 계산
3. OOB MSE가 가장 작은 조합 선택 (동률이면 grid 순서상 먼저 나온 조합)
4. 선택된 조합의 RF가 그대로 최종 모델 (별도 재학습 없음)

24개 후보 모두 같은 키 `make_random_state(s, 'common_training')`로 **새 객체**를 만들어 넘긴다.

**OOB 예측이 없는 관측치 처리**: sklearn은 OOB 트리가 하나도 없는 관측치에 NaN을 주지 않고, 예측값을 0으로 채운 뒤 `UserWarning("Some inputs do not have OOB scores...")`만 낸다. 그대로 두면 OOB MSE가 조용히 틀어지므로, 이 경고를 **오류로 바꿔** 실행을 멈추게 한다.

먼저 함수 없이 한 번 풀어서 실행한다.

In [7]:
import warnings

# OOB 예측이 없는 관측치가 생기면 경고 대신 오류로 멈춘다
warnings.filterwarnings('error', message='Some inputs do not have OOB scores')

# RF를 n_jobs=-1로 병렬 학습할 때 sklearn·joblib 사이에서 나오는 안내성 경고 (결과와 무관) → 끈다
warnings.filterwarnings('ignore', message='`sklearn.utils.parallel.delayed` should be used')

# OOB grid search로 고를 후보 값들 → 4 x 2 x 3 = 24개 조합
GRID_MIN_SAMPLES_LEAF = [5, 10, 20, 40]
GRID_MAX_FEATURES = [0.5, 1.0]
GRID_MAX_SAMPLES = [0.5, 0.7, 1.0]

def fit_rf_oob(X, y, scenario, stage, r=0, b=0, fold=0):
    # Fit = OOB grid search + 학습. 선택된 RF, 선택 조합, 조합별 OOB MSE 표를 돌려준다.
    rows = []
    best_model = None
    best_mse = np.inf
    best_params = None

    for leaf in GRID_MIN_SAMPLES_LEAF:
        for feat in GRID_MAX_FEATURES:
            for samp in GRID_MAX_SAMPLES:
                rf = RandomForestRegressor(
                    n_estimators=N_TREES,
                    min_samples_leaf=leaf,
                    max_features=feat,
                    max_samples=samp,
                    max_depth=MAX_DEPTH,
                    bootstrap=True,
                    oob_score=True,
                    n_jobs=-1,
                    random_state=make_random_state(scenario, stage, r=r, b=b, fold=fold),
                )
                rf.fit(X, y)

                oob_mse = np.mean((y - rf.oob_prediction_) ** 2)
                rows.append([leaf, feat, samp, oob_mse])

                if oob_mse < best_mse:
                    best_mse = oob_mse
                    best_model = rf
                    best_params = (leaf, feat, samp)

    # 예측은 단일 스레드로: n_jobs=-1로 예측하면 여러 스레드가 트리 예측을 더하는 순서가 실행마다 달라
    # 끝자리(1e-15 수준)가 흔들린다. 학습은 병렬 그대로, 예측만 n_jobs=1로 바꿔 같은 키 → 비트 단위로 같은 결과
    best_model.set_params(n_jobs=1)

    oob_table = pd.DataFrame(rows, columns=['min_samples_leaf', 'max_features', 'max_samples', 'oob_mse'])
    return best_model, best_params, oob_table


best_model, best_params, oob_table = fit_rf_oob(X_train, Y_train, "linear_homo_gaussian", 'common_training')

print("best model : ", best_model)
print("best params : ", best_params)
print("oob table : ")
pd.DataFrame(oob_table)

best model :  RandomForestRegressor(max_features=0.5, max_samples=1.0, min_samples_leaf=20,
                      n_estimators=30, n_jobs=1, oob_score=True,
                      random_state=RandomState(MT19937) at 0x243B285CA40)
best params :  (20, 0.5, 1.0)
oob table : 


,min_samples_leaf,max_features,max_samples,oob_mse
0,5,0.5,0.5,1.055068
1,5,0.5,0.7,1.073303
2,5,0.5,1.0,1.074798
3,5,1.0,0.5,1.076385
4,5,1.0,0.7,1.086976
5,5,1.0,1.0,1.098960
6,10,0.5,0.5,1.046688
7,10,0.5,0.7,1.051133
8,10,0.5,1.0,1.065368
9,10,1.0,0.5,1.062055


## 8. CV 잔차분산 $\widehat\sigma^2$ — 10-fold, 하이퍼파라미터 고정

관측값 $j$가 속한 fold를 $k(j)$라 하면

$$
r_j^{CV}=Y_j^{train}-\widehat f^{(-k(j))}(X_j^{train}),\qquad
\widehat\sigma^2=\frac1{n_{train}}\sum_{j=1}^{n_{train}}(r_j^{CV})^2
$$

- 분할: `KFold(10, shuffle=True, random_state=make_random_state(s, 'cv_split'))`
- 각 fold 모델 $\widehat f^{(-k)}$은 **튜닝하지 않는다.** §7에서 고른 `best_params`를 고정하고, fold를 뺀 900개로 RF **하나만** 학습한다 (키 `cv_model, fold=k`).
- $\widehat\sigma^2$는 OOF 잔차의 **제곱평균**이다 (`np.var` 아님).
- 한계
  - 잔차에는 평균함수 추정오차도 들어 있으므로 순수 오차분산의 불편추정량은 아니다.
  - `best_params`는 training 1000개 전체(검증 fold 포함)로 골랐으므로 약한 정보 누출로 $\widehat\sigma^2$가 아주 약간 작게 나올 수 있다 (후보가 24개뿐이라 영향은 작다고 본다).
- 진단: 기준 RF의 OOB 잔차 제곱평균 `sigma2_oob`를 옆에 출력한다. 24개 중 최솟값을 고른 값이라 작게 치우칠 수 있어 MC에는 쓰지 않는다.

In [8]:
leaf, feat, samp = best_params   # §7에서 고른 조합을 그대로 쓴다
print('고정한 조합 (min_samples_leaf, max_features, max_samples):', best_params)

kfold = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=make_random_state("linear_homo_gaussian", 'cv_split'))

oof_pred = np.full(N_TRAIN, np.nan)        # 각 관측치의 OOF 예측
oof_count = np.zeros(N_TRAIN, dtype=int)   # 각 관측치가 몇 번 예측됐는지

fold = 0
for train_idx, valid_idx in kfold.split(X_train):
    fold += 1   # fold는 1부터

    # 튜닝 없이 best_params로 RF 하나만 학습
    fold_model = RandomForestRegressor(
        n_estimators=N_TREES,
        min_samples_leaf=leaf,
        max_features=feat,
        max_samples=samp,
        max_depth=MAX_DEPTH,
        bootstrap=True,
        n_jobs=-1,
        random_state=make_random_state("linear_homo_gaussian", 'cv_model', fold=fold),
    )
    fold_model.fit(X_train[train_idx], Y_train[train_idx])
    fold_model.set_params(n_jobs=1)   # 예측은 단일 스레드 (합산 순서 고정 → 재현)

    oof_pred[valid_idx] = fold_model.predict(X_train[valid_idx])
    oof_count[valid_idx] += 1

# 모든 관측치가 자기가 빠진 fold 모델로 정확히 한 번 예측됐는지
print('예측 횟수 최소·최대 :', oof_count.min(), oof_count.max(), '  ← 둘 다 1이어야 함')
assert (oof_count == 1).all()

cv_residual = Y_train - oof_pred
sigma2_hat = np.mean(cv_residual ** 2)     # 제곱평균
if not np.isfinite(sigma2_hat):
    raise ValueError('sigma2_hat이 유한하지 않음')
if sigma2_hat == 0:
    raise ValueError('sigma2_hat = 0 (퇴화)')
sigma_hat = np.sqrt(sigma2_hat)

# 진단: 기준 RF의 OOB 잔차 제곱평균 (MC에는 쓰지 않음)
sigma2_oob = np.mean((Y_train - best_model.oob_prediction_) ** 2)

print()
print('sigma2_hat (CV, MC에 사용) :', sigma2_hat)
print('sigma_hat                  :', sigma_hat)
print('pooled CV RMSE             :', np.sqrt(sigma2_hat))
print('sigma2_oob (진단용)        :', sigma2_oob)
print('참 오차분산                :', 1.0)

고정한 조합 (min_samples_leaf, max_features, max_samples): (20, 0.5, 1.0)


예측 횟수 최소·최대 : 1 1   ← 둘 다 1이어야 함

sigma2_hat (CV, MC에 사용) : 1.0637955954459866
sigma_hat                  : 1.0314046710413844
pooled CV RMSE             : 1.0314046710413844
sigma2_oob (진단용)        : 1.044270307881376
참 오차분산                : 1.0


In [9]:
# 진단: grid 25개에서 기준 RF 예측 f̂(x)와 참평균 m(x)의 차이 (통과 기준이 아니라 결과 해석용)
# CP·MC 모두 f̂를 중심으로 하므로, f̂가 크게 빗나가는 grid에서는 coverage가 목표에서 벗어날 수 있다
base_grid_pred = best_model.predict(X_grid)
grid_true_mean = mean_function("linear_homo_gaussian", X_grid)
grid_error = base_grid_pred - grid_true_mean

grid_rmse = np.sqrt(np.mean(grid_error ** 2))
grid_mae = np.mean(np.abs(grid_error))
print("grid RMSE (f̂ vs m) :", grid_rmse)
print("grid MAE  (f̂ vs m) :", grid_mae)

pd.DataFrame({
    "x1" : X_grid[:, 0],
    "x2" : X_grid[:, 1],
    "true_mean" : grid_true_mean,
    "base_prediction" : base_grid_pred,
    "error" : grid_error,
})

grid RMSE (f̂ vs m) : 0.2195383820076642
grid MAE  (f̂ vs m) : 0.1649741762794429


,x1,x2,true_mean,base_prediction,error
0,-1.0,-1.0,-2.0,-1.529330,0.470670
1,-1.0,-0.5,-1.5,-1.385685,0.114315
2,-1.0,0.0,-1.0,-1.112744,-0.112744
3,-1.0,0.5,-0.5,-0.356248,0.143752
4,-1.0,1.0,0.0,0.467809,0.467809
5,-0.5,-1.0,-1.5,-1.368347,0.131653
6,-0.5,-0.5,-1.0,-0.906854,0.093146
7,-0.5,0.0,-0.5,-0.519003,-0.019003
8,-0.5,0.5,0.0,0.073715,0.073715
9,-0.5,1.0,0.5,0.896632,0.396632


## 9. 한 번의 반복 r = 1 따라가기

### 9-1. 평가 반응값 (채점용 정답 Y)

$$
Y_i^{grid,(r)} = m(\underline x_i) + \varepsilon_i^{grid,(r)},\qquad \varepsilon\sim\text{true DGP 오차}
$$

- 키 `test_error, r`에서 25개를 한 번에 뽑는다.
- 구간을 만드는 데는 쓰지 않고, MC·CP와 두 alpha의 포함 여부 채점에만 **공통으로** 쓴다.

In [10]:
random_seed = make_random_state("linear_homo_gaussian", "test_error", r = 1)
epsilon_test = draw_error("linear_homo_gaussian", random_seed, N_GRID)
Y_test = mean_function("linear_homo_gaussian", X_grid) + epsilon_test

test_data = pd.DataFrame({
    "x1" : X_grid[:,0],
    "x2" : X_grid[:, 1],
    "y" : Y_test
})

test_data

,x1,x2,y
0,-1.0,-1.0,-1.741305
1,-1.0,-0.5,-1.095319
2,-1.0,0.0,-0.037627
3,-1.0,0.5,-1.590321
4,-1.0,1.0,0.638909
5,-0.5,-1.0,-2.373521
6,-0.5,-0.5,-0.288985
7,-0.5,0.0,0.043390
8,-0.5,0.5,-0.302299
9,-0.5,1.0,-0.394149


### 9-2. CP — calibration과 conformal quantile

매 r마다 true DGP에서 calibration 1000개를 새로 만든다 (키 `cp_calibration, r`, 생성기 하나에서 X 먼저, 이어서 오차). 기준 RF $\widehat f$는 고정.

$$
S_a=|Y_a^{cal}-\widehat f(X_a^{cal})|,\qquad
k_\alpha=\lceil(n_{cal}+1)(1-\alpha)\rceil,\qquad
\widehat q_\alpha=S_{(k_\alpha)}
$$

- 정렬한 score에서 $k$번째 값 = 0-based index `k-1` (보간 없음)
- $k>n_{cal}$이면 $\widehat q=\infty$
- $n_{cal}=1000$: $\alpha=0.10$ → 901번째, $\alpha=0.05$ → 951번째

In [11]:
random_seed = make_random_state('linear_homo_gaussian', 'cp_calibration', r=1)
CP_X = random_seed.uniform(-1, 1, size=(N_CAL, 2))
CP_eps = draw_error('linear_homo_gaussian', random_seed, N_CAL)
CP_Y = mean_function('linear_homo_gaussian', CP_X) + CP_eps

cp_predict = best_model.predict(CP_X)
s = sorted(np.abs(CP_Y - cp_predict))

k_alphas = {}
q_alphas = {}
for alpha in ALPHAS:
    k_alpha = np.ceil((len(CP_X) + 1) * (1 - alpha))
    if k_alpha > N_CAL:
        q_alpha = np.inf
    else:
        q_alpha = s[int(k_alpha) - 1]
    k_alphas[alpha] = k_alpha      # append 대신 키로 저장
    q_alphas[alpha] = q_alpha

display(k_alphas, q_alphas)

{0.1: np.float64(901.0), 0.05: np.float64(951.0)}

{0.1: np.float64(1.6947398511543839), 0.05: np.float64(1.929899210912544)}

In [12]:
grid_predict = best_model.predict(X_grid)

CP_interval = pd.DataFrame({
    "x_1" : X_grid[:,0],
    "x_2" : X_grid[:,1],
    "predict" : grid_predict[:],
    "lower_0.1"  : grid_predict - q_alphas[0.10],
    "upper_0.1"  : grid_predict + q_alphas[0.10],
    "lower_0.05" : grid_predict - q_alphas[0.05],
    "upper_0.05" : grid_predict + q_alphas[0.05],

})
CP_interval

,x_1,x_2,predict,lower_0.1,upper_0.1,lower_0.05,upper_0.05
0,-1.0,-1.0,-1.529330,-3.224070,0.165410,-3.459229,0.400569
1,-1.0,-0.5,-1.385685,-3.080425,0.309055,-3.315584,0.544214
2,-1.0,0.0,-1.112744,-2.807484,0.581996,-3.042643,0.817155
3,-1.0,0.5,-0.356248,-2.050988,1.338492,-2.286148,1.573651
4,-1.0,1.0,0.467809,-1.226930,2.162549,-1.462090,2.397709
5,-0.5,-1.0,-1.368347,-3.063087,0.326393,-3.298246,0.561552
6,-0.5,-0.5,-0.906854,-2.601594,0.787885,-2.836754,1.023045
7,-0.5,0.0,-0.519003,-2.213743,1.175737,-2.448903,1.410896
8,-0.5,0.5,0.073715,-1.621025,1.768455,-1.856184,2.003614
9,-0.5,1.0,0.896632,-0.798108,2.591372,-1.033267,2.826531


### 9-3. MC — 가상 데이터 하나(b = 1) 따라가기

1. 원래 training X는 그대로 두고 가상 반응값을 만든다 (키 `mc_dataset, r, b`):
$$Y_j^{*(r,b)}=\widehat f(X_j^{train})+\epsilon_j^{*},\qquad \epsilon_j^{*}\sim N(0,\widehat\sigma^2)$$
2. 이 가상 데이터로 **RF 전체를 새로** Fit (OOB 튜닝 포함, 키 `mc_training, r, b`)
3. grid 25개를 한 번에 예측하고, 새 관측오차를 더한다 (키 `mc_new_error, r, b`):
$$\widetilde Y_i^{*(r,b)}=\widehat f^{*(r,b)}(\underline x_i)+\epsilon_{new,i}^{*},\qquad \epsilon_{new,i}^{*}\sim N(0,\widehat\sigma^2)$$

Student-t scenario에서도 MC 오차는 Gaussian이다.

In [13]:
random_seed = make_random_state("linear_homo_gaussian", "mc_dataset", r = 1, b = 1)
mc_epsilon = random_seed.normal(0, sigma_hat, N_TRAIN)
mc_Y = best_model.predict(X_train) + mc_epsilon

mc_data = pd.DataFrame({
    "x1" : X_train[:,0],
    "x2" : X_train[:,1],
    "y" : mc_Y
})

mc_data

,x1,x2,y
0,0.704293,0.557416,0.719146
1,-0.378332,-0.471905,-0.648668
2,0.098369,0.711712,0.116681
3,0.764764,0.576740,-0.326676
4,-0.073880,-0.037704,0.776585
...,...,...,...
995,-0.992051,0.197562,0.021842
996,-0.828187,0.348705,1.215649
997,-0.910988,0.050766,-0.887887
998,0.173330,-0.007109,0.016144


In [14]:
mc_model, mc_params, mc_table = fit_rf_oob(X_train, mc_Y, "linear_homo_gaussian", "mc_training", r = 1, b = 1)
print("mc_model :", mc_model)
print("mc_params :", mc_params)
print("mc_table :")
display(mc_table)

mc_model : RandomForestRegressor(max_samples=0.5, min_samples_leaf=20, n_estimators=30,
                      n_jobs=1, oob_score=True,
                      random_state=RandomState(MT19937) at 0x243B285DE40)
mc_params : (20, 1.0, 0.5)
mc_table :


,min_samples_leaf,max_features,max_samples,oob_mse
0,5,0.5,0.5,1.168835
1,5,0.5,0.7,1.174923
2,5,0.5,1.0,1.202141
3,5,1.0,0.5,1.201943
4,5,1.0,0.7,1.220795
5,5,1.0,1.0,1.237889
6,10,0.5,0.5,1.153547
7,10,0.5,0.7,1.150660
8,10,0.5,1.0,1.162806
9,10,1.0,0.5,1.159787


In [15]:
random_seed = make_random_state("linear_homo_gaussian", "mc_new_error", r = 1, b = 1)
mc_new_epsilon = random_seed.normal(0, sigma_hat, N_GRID)
Y_tilde = mc_model.predict(X_grid) + mc_new_epsilon

### 9-4. MC — b = 1..B 반복하고 분위수 구간 만들기

9-3의 세 단계를 b = 1..B로 반복해 `mc_samples`를 채운다 (smoke B=5, full B=200). 행은 재학습 모델 b, 열은 grid i → shape `(B, 25)`.

$$
C_{MC,\alpha}(\underline x_i)=[Q_{\alpha/2}(\widetilde Y_i^{*(1:B)}),\ Q_{1-\alpha/2}(\widetilde Y_i^{*(1:B)})]
$$

- 분위수는 **열마다(B 축, `axis=0`)** 계산한다. 다른 grid의 값을 섞지 않는다.
- 같은 B×25 표본을 두 alpha에 같이 쓴다.
- 예측이 끝난 RF는 버린다 (B×25 배열만 유지).

In [16]:
best_train_pred = best_model.predict(X_train)   # 가상 반응값의 중심 f̂(X_train), b와 무관하므로 한 번만

mc_samples = np.zeros((B, N_GRID))   # 행 = b, 열 = grid

for b in range(1, B + 1):
    # 1. 가상 training 반응값 Y* = f̂(X_train) + N(0, σ̂²)
    random_seed = make_random_state("linear_homo_gaussian", "mc_dataset", r = 1, b = b)
    mc_Y = best_train_pred + random_seed.normal(0, sigma_hat, N_TRAIN)

    # 2. 가상 데이터로 OOB 튜닝 + 학습
    mc_model, mc_params, mc_table = fit_rf_oob(X_train, mc_Y, "linear_homo_gaussian", "mc_training", r = 1, b = b)

    # 3. grid 25개 예측 + 새 관측오차 N(0, σ̂²)
    random_seed = make_random_state("linear_homo_gaussian", "mc_new_error", r = 1, b = b)
    mc_samples[b - 1] = mc_model.predict(X_grid) + random_seed.normal(0, sigma_hat, N_GRID)

    print(f"b={b}/{B}  선택된 조합 {mc_params}")
    del mc_model   # 예측이 끝난 RF는 버린다

print()
print("mc_samples.shape :", mc_samples.shape)
print("b=1 행이 9-3의 Y_tilde와 같은지 :", np.allclose(mc_samples[0], Y_tilde))

# 열마다(B 축) 분위수 → 두 alpha의 MC 구간
mc_lower = {}
mc_upper = {}
for alpha in ALPHAS:
    mc_lower[alpha] = np.quantile(mc_samples, alpha / 2, axis=0, method=MC_QUANTILE_METHOD)
    mc_upper[alpha] = np.quantile(mc_samples, 1 - alpha / 2, axis=0, method=MC_QUANTILE_METHOD)

MC_interval = pd.DataFrame({
    "x_1" : X_grid[:,0],
    "x_2" : X_grid[:,1],
    "lower_0.1"  : mc_lower[0.10],
    "upper_0.1"  : mc_upper[0.10],
    "lower_0.05" : mc_lower[0.05],
    "upper_0.05" : mc_upper[0.05],
})
MC_interval

b=1/5  선택된 조합 (20, 1.0, 0.5)


b=2/5  선택된 조합 (20, 1.0, 0.5)


b=3/5  선택된 조합 (10, 0.5, 0.7)


b=4/5  선택된 조합 (20, 0.5, 1.0)


b=5/5  선택된 조합 (20, 0.5, 0.5)

mc_samples.shape : (5, 25)
b=1 행이 9-3의 Y_tilde와 같은지 : True


,x_1,x_2,lower_0.1,upper_0.1,lower_0.05,upper_0.05
0,-1.0,-1.0,-2.451802,-0.522211,-2.453234,-0.482235
1,-1.0,-0.5,-2.651051,0.138833,-2.827147,0.169027
2,-1.0,0.0,-1.476615,0.168924,-1.487177,0.313052
3,-1.0,0.5,-2.162298,1.139718,-2.323164,1.178210
4,-1.0,1.0,-0.008053,1.124358,-0.022042,1.175449
5,-0.5,-1.0,-1.956591,0.477402,-2.013706,0.552984
6,-0.5,-0.5,-3.236810,0.068999,-3.285331,0.152720
7,-0.5,0.0,-1.788184,-0.266128,-1.806428,-0.260565
8,-0.5,0.5,-1.903596,0.703763,-2.005364,0.833542
9,-0.5,1.0,0.447544,2.072840,0.422440,2.178543


In [17]:
# grid 점마다 B개의 MC 가상 반응값 Ỹ*(b) 전체 보기 (행 = grid 25개, 열 = b)
mc_samples_table = pd.DataFrame({
    "x_1" : X_grid[:,0],
    "x_2" : X_grid[:,1],
})
for b in range(1, B + 1):
    mc_samples_table[f"b={b}"] = mc_samples[b - 1]   # mc_samples의 b번째 행 = 모델 b가 만든 25개 값

mc_samples_table

,x_1,x_2,b=1,b=2,b=3,b=4,b=5
0,-1.0,-1.0,-0.442259,-2.440348,-0.842022,-2.454666,-1.823623
1,-1.0,-0.5,-0.825392,0.199222,-3.003243,-0.102725,-1.242284
2,-1.0,0.0,-1.497739,-0.984101,-1.243498,-1.392120,0.457180
3,-1.0,0.5,-0.466933,-2.484030,0.831782,-0.875373,1.216702
4,-1.0,1.0,-0.036031,0.610174,0.103857,0.715627,1.226540
5,-0.5,-1.0,-1.499667,-0.726769,-2.070822,-0.127250,0.628565
6,-0.5,-0.5,-0.600771,0.236441,-2.848642,-3.333852,-2.044530
7,-0.5,0.0,-1.824672,-0.422290,-0.255001,-1.642232,-0.310636
8,-0.5,0.5,-2.107132,-1.089454,-0.726317,0.963321,-0.334470
9,-0.5,1.0,0.648375,2.284247,1.227212,0.397336,1.153685


### 9-5. 평가 — 포함 여부와 구간 길이

9-1에서 만든 평가 반응값 `Y_test` **하나**로 MC·CP와 두 alpha의 구간을 모두 채점한다.

$$
I_{h,i,r,\alpha}=\mathbf 1\{L_{h,i,r,\alpha}\le Y_i^{grid,(r)}\le U_{h,i,r,\alpha}\},
\qquad
\ell_{h,i,r,\alpha}=U_{h,i,r,\alpha}-L_{h,i,r,\alpha}
$$

- 경계 포함(`lower <= Y_test <= upper`)으로 판정한다.
- 배열끼리 비교하므로 grid 25개가 각자 따로 0/1로 판정된다 (`covered`는 길이 25의 True/False 배열).
- 지금은 r = 1 하나라서 count는 0 또는 1이다. r = 1..R로 반복하면 grid마다 count가 쌓이고, 아래가 최종 지표가 된다.

$$
\widehat c_{h,\alpha}(\underline x_i)=\frac{\text{count}}{R},\qquad
\overline\ell_{h,\alpha}(\underline x_i)=\frac1R\sum_{r}\ell_{h,i,r,\alpha}
$$

- 길이는 포함 여부와 관계없이 모든 반복의 값을 평균한다.
- 평균 하한·상한으로 만든 구간의 포함 여부로 coverage를 대체하지 않는다.
- smoke(B=5)의 결과는 구조 확인용이며 연구 결과가 아니다.

In [18]:
# MC 평가: grid 25개를 한 번에 채점 (r = 1)
MC_eval = pd.DataFrame({
    "x1" : X_grid[:, 0],
    "x2" : X_grid[:, 1],
    "y_test" : Y_test,
})

for alpha in ALPHAS:
    covered = (mc_lower[alpha] <= Y_test) & (Y_test <= mc_upper[alpha])   # 25개 True/False (경계 포함)
    MC_eval[f"{alpha}_count"] = covered.astype(int)                       # 지금은 r=1이라 0 또는 1
    MC_eval[f"{alpha}_length"] = mc_upper[alpha] - mc_lower[alpha]

for alpha in ALPHAS:
    print(f"alpha={alpha}: 포함된 grid {MC_eval[f'{alpha}_count'].sum()} / {N_GRID}")
MC_eval

alpha=0.1: 포함된 grid 18 / 25
alpha=0.05: 포함된 grid 18 / 25


,x1,x2,y_test,0.1_count,0.1_length,0.05_count,0.05_length
0,-1.0,-1.0,-1.741305,1,1.929591,1,1.970999
1,-1.0,-0.5,-1.095319,1,2.789884,1,2.996174
2,-1.0,0.0,-0.037627,1,1.645540,1,1.800230
3,-1.0,0.5,-1.590321,1,3.302016,1,3.501374
4,-1.0,1.0,0.638909,1,1.132411,1,1.197491
5,-0.5,-1.0,-2.373521,0,2.433993,0,2.566690
6,-0.5,-0.5,-0.288985,1,3.305809,1,3.438051
7,-0.5,0.0,0.043390,0,1.522056,0,1.545863
8,-0.5,0.5,-0.302299,1,2.607359,1,2.838906
9,-0.5,1.0,-0.394149,0,1.625296,0,1.756104


In [19]:
CP_eval = pd.DataFrame({
    "x1" : X_grid[:, 0],
    "x2" : X_grid[:, 1],
    "y_test" : Y_test,
})

for alpha in ALPHAS:
    covered = (grid_predict - q_alphas[alpha] <= Y_test) & (Y_test <= grid_predict + q_alphas[alpha])   # 25개 True/False (경계 포함)
    CP_eval[f"{alpha}_count"] = covered.astype(int)                       # 지금은 r=1이라 0 또는 1
    CP_eval[f"{alpha}_length"] = 2 * q_alphas[alpha]

for alpha in ALPHAS:
    print(f"alpha={alpha}: 포함된 grid {CP_eval[f'{alpha}_count'].sum()} / {N_GRID}")
CP_eval

alpha=0.1: 포함된 grid 23 / 25
alpha=0.05: 포함된 grid 23 / 25


,x1,x2,y_test,0.1_count,0.1_length,0.05_count,0.05_length
0,-1.0,-1.0,-1.741305,1,3.38948,1,3.859798
1,-1.0,-0.5,-1.095319,1,3.38948,1,3.859798
2,-1.0,0.0,-0.037627,1,3.38948,1,3.859798
3,-1.0,0.5,-1.590321,1,3.38948,1,3.859798
4,-1.0,1.0,0.638909,1,3.38948,1,3.859798
5,-0.5,-1.0,-2.373521,1,3.38948,1,3.859798
6,-0.5,-0.5,-0.288985,1,3.38948,1,3.859798
7,-0.5,0.0,0.043390,1,3.38948,1,3.859798
8,-0.5,0.5,-0.302299,1,3.38948,1,3.859798
9,-0.5,1.0,-0.394149,1,3.38948,1,3.859798


## 10. r 반복 — conditional coverage

9-1 ~ 9-5(평가 Y → CP → MC B개 → 채점)를 `run_one_r(scenario, r)` 함수로 묶고 r = 1..R로 반복한다. r = 1로 고정돼 있던 난수 키는 모두 `r = r`로 바꾼다.

고정: training, 기준 RF `best_model`, `sigma_hat`, grid. 매 r 새로 생성: 평가 Y, calibration, MC 가상 데이터·RF·새 오차.

함수는 한 r의 결과를 `interval_metrics.csv`와 같은 모양(한 행 = grid × alpha × method, 100행)으로 돌려준다. R번 쌓은 뒤 **grid마다** 반복 방향으로 평균하면 conditional coverage가 된다.

$$
\widehat c_{h,\alpha}(\underline x_i)=\frac1R\sum_{r=1}^{R} I_{h,i,r,\alpha},\qquad
\overline\ell_{h,\alpha}(\underline x_i)=\frac1R\sum_{r=1}^{R}\ell_{h,i,r,\alpha}
$$

grid 25개를 가로질러 합친 숫자(예: "22/25")는 conditional coverage가 아니므로 만들지 않는다. smoke(R=2)에서는 coverage가 0, 0.5, 1 중 하나로만 나오며 구조 확인용이다.

In [20]:
def run_one_r(scenario, r):
    # 한 번의 반복 r: 평가 Y → CP 구간 → MC 구간 → 채점.
    # 돌려주는 것: (채점 결과 100행 = grid 25 × alpha 2 × method 2, MC 선택 조합 B행)
    # 고정값(best_model, best_train_pred, grid_predict, sigma_hat, X_train, X_grid)은 위에서 만든 것을 그대로 쓴다.

    # 1. 평가 반응값 (9-1)
    random_seed = make_random_state(scenario, "test_error", r = r)
    true_mean = mean_function(scenario, X_grid)
    Y_test = true_mean + draw_error(scenario, random_seed, N_GRID)

    # 2. CP (9-2): 새 calibration → score → 순위 분위수
    random_seed = make_random_state(scenario, "cp_calibration", r = r)
    CP_X = random_seed.uniform(-1, 1, size=(N_CAL, 2))
    CP_Y = mean_function(scenario, CP_X) + draw_error(scenario, random_seed, N_CAL)
    s = sorted(np.abs(CP_Y - best_model.predict(CP_X)))

    cp_lower = {}
    cp_upper = {}
    for alpha in ALPHAS:
        k_alpha = np.ceil((N_CAL + 1) * (1 - alpha))
        if k_alpha > N_CAL:
            q_alpha = np.inf
        else:
            q_alpha = s[int(k_alpha) - 1]
        cp_lower[alpha] = grid_predict - q_alpha
        cp_upper[alpha] = grid_predict + q_alpha

    # 3. MC (9-3, 9-4): b = 1..B 가상 데이터 → OOB 튜닝 + 학습 → grid 예측 + 새 오차
    mc_samples = np.zeros((B, N_GRID))
    selection_rows = []   # (r, b)마다 OOB 튜닝으로 선택된 조합 → model_selection.csv
    for b in range(1, B + 1):
        random_seed = make_random_state(scenario, "mc_dataset", r = r, b = b)
        mc_Y = best_train_pred + random_seed.normal(0, sigma_hat, N_TRAIN)

        mc_model, mc_params, mc_table = fit_rf_oob(X_train, mc_Y, scenario, "mc_training", r = r, b = b)
        selection_rows.append([scenario, "mc", r, b, mc_params[0], mc_params[1], mc_params[2], mc_table["oob_mse"].min()])

        random_seed = make_random_state(scenario, "mc_new_error", r = r, b = b)
        mc_samples[b - 1] = mc_model.predict(X_grid) + random_seed.normal(0, sigma_hat, N_GRID)
        del mc_model

    mc_lower = {}
    mc_upper = {}
    for alpha in ALPHAS:
        mc_lower[alpha] = np.quantile(mc_samples, alpha / 2, axis=0, method=MC_QUANTILE_METHOD)
        mc_upper[alpha] = np.quantile(mc_samples, 1 - alpha / 2, axis=0, method=MC_QUANTILE_METHOD)

    # 4. 채점 (9-5): 같은 Y_test로 두 방법·두 alpha
    tables = []
    for alpha in ALPHAS:
        for method in ["mc", "cp"]:
            if method == "mc":
                lower = mc_lower[alpha]
                upper = mc_upper[alpha]
            else:
                lower = cp_lower[alpha]
                upper = cp_upper[alpha]

            covered = (lower <= Y_test) & (Y_test <= upper)   # 25개 True/False (경계 포함)
            tables.append(pd.DataFrame({
                "scenario" : scenario,
                "r" : r,
                "grid_id" : np.arange(N_GRID),
                "x1" : X_grid[:, 0],
                "x2" : X_grid[:, 1],
                "alpha" : alpha,
                "method" : method,
                "y_true" : Y_test,
                "true_mean" : true_mean,
                "lower" : lower,
                "upper" : upper,
                "covered" : covered.astype(int),
                "length" : upper - lower,
            }))

    r_selection = pd.DataFrame(selection_rows, columns=[
        "scenario", "stage", "r", "b", "min_samples_leaf", "max_features", "max_samples", "oob_mse",
    ])
    return pd.concat(tables, ignore_index=True), r_selection

In [21]:
# r = 1..R 반복 (지금은 linear_homo_gaussian 하나, 고정값은 위 6~8절에서 만든 것)
best_train_pred = best_model.predict(X_train)
grid_predict = best_model.predict(X_grid)

r_tables = []
selection_tables = []
for r in range(1, R + 1):
    r_result, r_selection = run_one_r("linear_homo_gaussian", r)
    r_tables.append(r_result)
    selection_tables.append(r_selection)
    print(f"r={r}/{R} 완료")

interval_metrics = pd.concat(r_tables, ignore_index=True)

# 같은 r·grid·alpha 안에서 mc, cp가 번갈아 나오도록 정렬 (alpha는 0.10 → 0.05)
interval_metrics["method_order"] = interval_metrics["method"].map({"mc": 0, "cp": 1})
interval_metrics = interval_metrics.sort_values(
    ["r", "grid_id", "alpha", "method_order"], ascending=[True, True, False, True]
).drop(columns="method_order").reset_index(drop=True)
display(interval_metrics)
print()
print("interval_metrics 행 수 :", len(interval_metrics), f"  ← R x 100 = {R * 100}")

# r=1 결과가 9-5에서 한 r=1 채점과 같은지 (같은 키 → 같은 결과)
r1 = interval_metrics[interval_metrics["r"] == 1]
print("r=1 MC 포함 수가 9-5와 같은지 :", (r1[(r1["method"] == "mc") & (r1["alpha"] == 0.10)]["covered"].values == MC_eval["0.1_count"].values).all())
print("r=1 CP 포함 수가 9-5와 같은지 :", (r1[(r1["method"] == "cp") & (r1["alpha"] == 0.10)]["covered"].values == CP_eval["0.1_count"].values).all())

r=1/2 완료


r=2/2 완료


,scenario,r,grid_id,x1,x2,alpha,method,y_true,true_mean,lower,upper,covered,length
0,linear_homo_gaussian,1,0,-1.0,-1.0,0.10,mc,-1.741305,-2.0,-2.451802,-0.522211,1,1.929591
1,linear_homo_gaussian,1,0,-1.0,-1.0,0.10,cp,-1.741305,-2.0,-3.224070,0.165410,1,3.389480
2,linear_homo_gaussian,1,0,-1.0,-1.0,0.05,mc,-1.741305,-2.0,-2.453234,-0.482235,1,1.970999
3,linear_homo_gaussian,1,0,-1.0,-1.0,0.05,cp,-1.741305,-2.0,-3.459229,0.400569,1,3.859798
4,linear_homo_gaussian,1,1,-1.0,-0.5,0.10,mc,-1.095319,-1.5,-2.651051,0.138833,1,2.789884
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,linear_homo_gaussian,2,23,1.0,0.5,0.05,cp,0.081565,1.5,-0.817277,3.229821,1,4.047098
196,linear_homo_gaussian,2,24,1.0,1.0,0.10,mc,3.209252,2.0,1.837260,2.617354,0,0.780094
197,linear_homo_gaussian,2,24,1.0,1.0,0.10,cp,3.209252,2.0,-0.043599,3.387973,1,3.431571
198,linear_homo_gaussian,2,24,1.0,1.0,0.05,mc,3.209252,2.0,1.814498,2.654266,0,0.839768



interval_metrics 행 수 : 200   ← R x 100 = 200
r=1 MC 포함 수가 9-5와 같은지 : True
r=1 CP 포함 수가 9-5와 같은지 : True


In [22]:
# grid마다 반복 방향(r)으로 평균 → conditional coverage와 평균 길이
point_summary = interval_metrics.groupby(["scenario", "grid_id", "x1", "x2", "alpha", "method"], as_index=False).agg(
    n_covered = ("covered", "sum"),
    n_evaluated = ("covered", "count"),
    coverage = ("covered", "mean"),
    length_mean = ("length", "mean"),
)

# interval_metrics와 같은 순서: grid → alpha(0.10 → 0.05) → method(mc → cp)
point_summary["method_order"] = point_summary["method"].map({"mc": 0, "cp": 1})
point_summary = point_summary.sort_values(
    ["grid_id", "alpha", "method_order"], ascending=[True, False, True]
).drop(columns="method_order").reset_index(drop=True)

print("point_summary 행 수 :", len(point_summary), "  ← 25 grid x 2 alpha x 2 method = 100")
print("n_evaluated (모든 행이 R이어야 함) :", point_summary["n_evaluated"].unique())

# 보기 쉽게: grid마다 한 행, 방법·alpha별 coverage와 평균 길이를 옆으로
coverage_view = point_summary.pivot_table(
    index=["grid_id", "x1", "x2"], columns=["method", "alpha"], values=["coverage", "length_mean"]
)
column_order = []
for value in ["coverage", "length_mean"]:
    for method in ["mc", "cp"]:
        for alpha in ALPHAS:          # ALPHAS = [0.10, 0.05]
            column_order.append((value, method, alpha))
coverage_view = coverage_view[column_order]
coverage_view

point_summary 행 수 : 100   ← 25 grid x 2 alpha x 2 method = 100
n_evaluated (모든 행이 R이어야 함) : [2]


coverage                length_mean                      \
method                  mc        cp               mc                  cp   
alpha                 0.10 0.05 0.10 0.05        0.10      0.05      0.10   
grid_id x1   x2                                                             
0       -1.0 -1.0      1.0  1.0  1.0  1.0    2.574398  2.769495  3.410525   
1       -1.0 -0.5      1.0  1.0  1.0  1.0    2.636209  2.845414  3.410525   
2       -1.0  0.0      0.5  0.5  1.0  1.0    1.490312  1.621500  3.410525   
3       -1.0  0.5      0.5  0.5  0.5  0.5    3.134133  3.298424  3.410525   
4       -1.0  1.0      1.0  1.0  1.0  1.0    1.427939  1.492966  3.410525   
5       -0.5 -1.0      0.0  0.0  0.5  1.0    2.562689  2.672648  3.410525   
6       -0.5 -0.5      1.0  1.0  1.0  1.0    2.523123  2.651858  3.410525   
7       -0.5  0.0      0.5  0.5  1.0  1.0    1.745426  1.835793  3.410525   
8       -0.5  0.5      1.0  1.0  1.0  1.0    2.526756  2.694225  3.410525   
9       -0.5  1.0      0.0  0.0  1.0  1.0    1.512776  1.644375  3.410525   
10       0.0 -1.0      1.0  1.0  1.0  1.0    2.351385  2.520862  3.410525   
11       0.0 -0.5      0.5  0.5  0.5  0.5    2.674547  2.791045  3.410525   
12       0.0  0.0      1.0  1.0  1.0  1.0    2.233042  2.300138  3.410525   
13       0.0  0.5      0.5  0.5  0.5  0.5    1.898317  2.019039  3.410525   
14       0.0  1.0      0.5  0.5  1.0  1.0    2.365143  2.514669  3.410525   
15       0.5 -1.0      1.0  1.0  1.0  1.0    3.275141  3.462155  3.410525   
16       0.5 -0.5      0.0  0.0  1.0  1.0    2.140930  2.306481  3.410525   
17       0.5  0.0      1.0  1.0  1.0  1.0    2.012463  2.180248  3.410525   
18       0.5  0.5      0.5  1.0  1.0  1.0    3.398520  3.530292  3.410525   
19       0.5  1.0      1.0  1.0  1.0  1.0    1.188504  1.263254  3.410525   
20       1.0 -1.0      1.0  1.0  0.5  0.5    2.681642  2.931459  3.410525   
21       1.0 -0.5      1.0  1.0  1.0  1.0    1.982837  2.086072  3.410525   
22       1.0  0.0      0.0  0.0  1.0  1.0    1.945643  2.135871  3.410525   
23       1.0  0.5      0.5  0.5  1.0  1.0    2.579648  2.702874  3.410525   
24       1.0  1.0      0.0  0.0  1.0  1.0    0.774316  0.824649  3.410525   

                             
method                       
alpha                  0.05  
grid_id x1   x2              
0       -1.0 -1.0  3.953448  
1       -1.0 -0.5  3.953448  
2       -1.0  0.0  3.953448  
3       -1.0  0.5  3.953448  
4       -1.0  1.0  3.953448  
5       -0.5 -1.0  3.953448  
6       -0.5 -0.5  3.953448  
7       -0.5  0.0  3.953448  
8       -0.5  0.5  3.953448  
9       -0.5  1.0  3.953448  
10       0.0 -1.0  3.953448  
11       0.0 -0.5  3.953448  
12       0.0  0.0  3.953448  
13       0.0  0.5  3.953448  
14       0.0  1.0  3.953448  
15       0.5 -1.0  3.953448  
16       0.5 -0.5  3.953448  
17       0.5  0.0  3.953448  
18       0.5  0.5  3.953448  
19       0.5  1.0  3.953448  
20       1.0 -1.0  3.953448  
21       1.0 -0.5  3.953448  
22       1.0  0.0  3.953448  
23       1.0  0.5  3.953448  
24       1.0  1.0  3.953448

## 11. 결과 저장

`results/check/<scenario>/` (full이면 `results/<scenario>/`) 아래에 저장한다. CSV는 `index=False`.

| 파일 | 내용 |
| --- | --- |
| `test_grid.csv` | grid 25개의 좌표·참평균·기준 RF 예측 |
| `interval_metrics.csv` | 반복 r × grid × alpha × method마다 한 행 (구간·포함 여부·길이) |
| `model_selection.csv` | OOB 튜닝으로 선택된 조합 (기준 RF 1행 + MC R×B행) |
| `point_summary.csv` | grid × alpha × method별 conditional coverage·평균 길이 |
| `method_comparison.csv` | grid × alpha별 MC·CP 나란히 + CP − MC 차이 |
| `config.json` | 설정·σ̂²·난수 정책·버전·완료 r 목록 |

노트북은 구현·확인용이라 실행할 때마다 같은 폴더를 덮어쓴다. 재개(완료된 r 건너뛰기)는 실제 실행용 `.py`에서 구현한다.

In [23]:
import json
import os
import platform

scenario = "linear_homo_gaussian"
save_dir = RESULTS_DIR / scenario
save_dir.mkdir(parents=True, exist_ok=True)
print("저장 위치 :", save_dir)

저장 위치 : C:\Users\cho03\Desktop\조민석\연구\results\check\linear_homo_gaussian


In [24]:
# test_grid.csv : grid 25개의 좌표·참평균·기준 RF 예측
test_grid = pd.DataFrame({
    "scenario" : scenario,
    "grid_id" : np.arange(N_GRID),
    "x1" : X_grid[:, 0],
    "x2" : X_grid[:, 1],
    "true_mean" : mean_function(scenario, X_grid),
    "base_prediction" : grid_predict,
})
test_grid.to_csv(save_dir / "test_grid.csv", index=False)
test_grid.head()

,scenario,grid_id,x1,x2,true_mean,base_prediction
0,linear_homo_gaussian,0,-1.0,-1.0,-2.0,-1.529330
1,linear_homo_gaussian,1,-1.0,-0.5,-1.5,-1.385685
2,linear_homo_gaussian,2,-1.0,0.0,-1.0,-1.112744
3,linear_homo_gaussian,3,-1.0,0.5,-0.5,-0.356248
4,linear_homo_gaussian,4,-1.0,1.0,0.0,0.467809


In [25]:
# interval_metrics.csv : §10에서 만든 원시 결과 그대로 (mc → cp 정렬)
interval_metrics.to_csv(save_dir / "interval_metrics.csv", index=False)
print("interval_metrics :", len(interval_metrics), "행")

interval_metrics : 200 행


In [26]:
# model_selection.csv : 기준 RF 1행 (r = b = 0) + MC R×B행
base_selection = pd.DataFrame([[
    scenario, "base", 0, 0, best_params[0], best_params[1], best_params[2], oob_table["oob_mse"].min()
]], columns=["scenario", "stage", "r", "b", "min_samples_leaf", "max_features", "max_samples", "oob_mse"])

model_selection = pd.concat([base_selection] + selection_tables, ignore_index=True)
model_selection.to_csv(save_dir / "model_selection.csv", index=False)
print("model_selection :", len(model_selection), f"행  ← 1 + R x B = {1 + R * B}")
model_selection

model_selection : 11 행  ← 1 + R x B = 11


,scenario,stage,r,b,min_samples_leaf,max_features,max_samples,oob_mse
0,linear_homo_gaussian,base,0,0,20,0.5,1.0,1.044270
1,linear_homo_gaussian,mc,1,1,20,1.0,0.5,1.137037
2,linear_homo_gaussian,mc,1,2,20,1.0,0.5,1.061394
3,linear_homo_gaussian,mc,1,3,10,0.5,0.7,1.021410
4,linear_homo_gaussian,mc,1,4,20,0.5,1.0,1.102278
5,linear_homo_gaussian,mc,1,5,20,0.5,0.5,1.160654
6,linear_homo_gaussian,mc,2,1,10,0.5,0.7,0.982485
7,linear_homo_gaussian,mc,2,2,20,0.5,1.0,1.164474
8,linear_homo_gaussian,mc,2,3,20,0.5,0.7,1.167169
9,linear_homo_gaussian,mc,2,4,20,1.0,0.5,1.081669


In [27]:
# point_summary.csv : §10에서 만든 grid × alpha × method별 conditional coverage·평균 길이
point_summary.to_csv(save_dir / "point_summary.csv", index=False)
print("point_summary :", len(point_summary), "행")

point_summary : 100 행


In [28]:
# method_comparison.csv : grid × alpha마다 MC·CP를 한 행에 나란히 놓고 CP − MC 차이 계산
mc_part = point_summary[point_summary["method"] == "mc"]
cp_part = point_summary[point_summary["method"] == "cp"]

mc_part = mc_part[["scenario", "grid_id", "x1", "x2", "alpha", "coverage", "length_mean"]].rename(
    columns={"coverage": "mc_coverage", "length_mean": "mc_length_mean"})
cp_part = cp_part[["scenario", "grid_id", "alpha", "coverage", "length_mean"]].rename(
    columns={"coverage": "cp_coverage", "length_mean": "cp_length_mean"})

method_comparison = mc_part.merge(cp_part, on=["scenario", "grid_id", "alpha"])
method_comparison["coverage_cp_minus_mc"] = method_comparison["cp_coverage"] - method_comparison["mc_coverage"]
method_comparison["length_cp_minus_mc"] = method_comparison["cp_length_mean"] - method_comparison["mc_length_mean"]

method_comparison = method_comparison[[
    "scenario", "grid_id", "x1", "x2", "alpha",
    "mc_coverage", "cp_coverage", "coverage_cp_minus_mc",
    "mc_length_mean", "cp_length_mean", "length_cp_minus_mc",
]]
method_comparison.to_csv(save_dir / "method_comparison.csv", index=False)
print("method_comparison :", len(method_comparison), "행  ← 25 grid x 2 alpha = 50")
method_comparison.head(10)

method_comparison : 50 행  ← 25 grid x 2 alpha = 50


,scenario,grid_id,x1,x2,alpha,mc_coverage,cp_coverage,coverage_cp_minus_mc,mc_length_mean,cp_length_mean,length_cp_minus_mc
0,linear_homo_gaussian,0,-1.0,-1.0,0.10,1.0,1.0,0.0,2.574398,3.410525,0.836127
1,linear_homo_gaussian,0,-1.0,-1.0,0.05,1.0,1.0,0.0,2.769495,3.953448,1.183954
2,linear_homo_gaussian,1,-1.0,-0.5,0.10,1.0,1.0,0.0,2.636209,3.410525,0.774316
3,linear_homo_gaussian,1,-1.0,-0.5,0.05,1.0,1.0,0.0,2.845414,3.953448,1.108034
4,linear_homo_gaussian,2,-1.0,0.0,0.10,0.5,1.0,0.5,1.490312,3.410525,1.920213
5,linear_homo_gaussian,2,-1.0,0.0,0.05,0.5,1.0,0.5,1.621500,3.953448,2.331948
6,linear_homo_gaussian,3,-1.0,0.5,0.10,0.5,0.5,0.0,3.134133,3.410525,0.276392
7,linear_homo_gaussian,3,-1.0,0.5,0.05,0.5,0.5,0.0,3.298424,3.953448,0.655024
8,linear_homo_gaussian,4,-1.0,1.0,0.10,1.0,1.0,0.0,1.427939,3.410525,1.982587
9,linear_homo_gaussian,4,-1.0,1.0,0.05,1.0,1.0,0.0,1.492966,3.953448,2.460482


In [29]:
# config.json : 이 결과를 만든 설정 전부
config = {}
config["design_doc"] = "시뮬.md"
config["run_mode"] = RUN_MODE
config["scenario"] = scenario

# 확정 설계값 (시뮬.md)
config["design"] = {
    "n_train": N_TRAIN,
    "n_cal": N_CAL,
    "grid_axis": GRID_AXIS,
    "n_grid": N_GRID,
    "alphas": ALPHAS,
    "R_full": 200,
    "B_full": 200,
    "variance_estimator": "cv",
    "mc_error_family": "gaussian",
}

# 이번 실행 규모 (smoke면 줄인 값)
config["run_size"] = {"R": R, "B": B, "n_trees": N_TREES}

# 구현 기본값 (변경 가능한 제안값)
config["implementation_defaults"] = {
    "master_seed": MASTER_SEED,
    "cv_folds": CV_FOLDS,
    "mc_quantile_method": MC_QUANTILE_METHOD,
    "rf_fixed": {"max_depth": MAX_DEPTH, "bootstrap": True, "oob_score": True, "n_jobs": -1},
    "rf_param_grid": {
        "min_samples_leaf": GRID_MIN_SAMPLES_LEAF,
        "max_features": GRID_MAX_FEATURES,
        "max_samples": GRID_MAX_SAMPLES,
    },
    "selection_rule": "OOB MSE = mean((y - oob_prediction_)^2) 최소, 동률이면 grid 순서상 먼저 나온 조합",
}

# 확정 방식 조합
config["method_choice"] = {
    "base_rf_tuning": "OOB",
    "sigma2_estimation": "CV, 기준 RF 조합 고정 (fold 안에서 다시 튜닝하지 않음)",
    "mc_rf_tuning": "OOB (가상 데이터마다 다시 튜닝)",
}

# 기준 RF와 분산 추정 결과
config["base_rf"] = {
    "best_params": {
        "min_samples_leaf": best_params[0],
        "max_features": best_params[1],
        "max_samples": best_params[2],
    },
    "oob_table": oob_table.to_dict(orient="records"),
}
config["sigma2_hat"] = float(sigma2_hat)
config["sigma2_oob_diagnostic"] = float(sigma2_oob)

# 진단값 (통과 기준 아님, 결과 해석용)
config["diagnostics"] = {
    "pooled_cv_rmse": float(sigma_hat),
    "grid_rmse_base_vs_true_mean": float(grid_rmse),
    "grid_mae_base_vs_true_mean": float(grid_mae),
}

# 난수 정책
config["rng"] = {
    "master_seed": MASTER_SEED,
    "scenario_ids": SCENARIO_IDS,
    "stage_ids": STAGE_IDS,
    "key_order": ["MASTER_SEED", "scenario_id", "stage_id", "r", "b", "fold"],
    "index_rules": "r=1..R, b=1..B, fold=1..CV_FOLDS; 해당 단계에 필요 없는 인덱스는 0",
    "generator": "np.random.RandomState(np.random.SeedSequence(key).generate_state(4)); 부를 때마다 새 객체",
}

# 실행 환경
config["library_versions"] = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit-learn": sklearn.__version__,
}
config["machine"] = {
    "platform": platform.platform(),
    "processor": platform.processor(),
    "cpu_count": os.cpu_count(),
}

# 완료 상태
completed_r = sorted(interval_metrics["r"].unique().tolist())
config["completed_r"] = completed_r
config["is_complete"] = len(completed_r) == R

with open(save_dir / "config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, ensure_ascii=False, indent=2, default=float)

print("config.json 저장 완료")
print("completed_r :", completed_r, " is_complete :", config["is_complete"])

config.json 저장 완료
completed_r : [1, 2]  is_complete : True


### 저장 확인

저장한 파일을 다시 읽어 행 수·key 중복·길이·집계 count를 확인한다.

In [30]:
saved_metrics = pd.read_csv(save_dir / "interval_metrics.csv")
saved_summary = pd.read_csv(save_dir / "point_summary.csv")
saved_comparison = pd.read_csv(save_dir / "method_comparison.csv")
saved_selection = pd.read_csv(save_dir / "model_selection.csv")
saved_grid = pd.read_csv(save_dir / "test_grid.csv")
with open(save_dir / "config.json", encoding="utf-8") as file:
    saved_config = json.load(file)

print("파일별 행 수")
print("  test_grid         :", len(saved_grid), "  ← 25")
print("  interval_metrics  :", len(saved_metrics), f"  ← R x 100 = {R * 100}")
print("  model_selection   :", len(saved_selection), f"  ← 1 + R x B = {1 + R * B}")
print("  point_summary     :", len(saved_summary), "  ← 100")
print("  method_comparison :", len(saved_comparison), "  ← 50")
print()

key = ["scenario", "r", "grid_id", "alpha", "method"]
print("interval_metrics key 중복 수 :", saved_metrics.duplicated(key).sum(), "  ← 0이어야 함")
print("length = upper - lower 최대 오차 :",
      (saved_metrics["length"] - (saved_metrics["upper"] - saved_metrics["lower"])).abs().max())
print("point_summary n_evaluated :", saved_summary["n_evaluated"].unique(), f"  ← 모두 R = {R}")
print("coverage_cp_minus_mc 최대 오차 :",
      (saved_comparison["coverage_cp_minus_mc"] - (saved_comparison["cp_coverage"] - saved_comparison["mc_coverage"])).abs().max())
print("config sigma2_hat :", saved_config["sigma2_hat"], " completed_r :", saved_config["completed_r"])

파일별 행 수
  test_grid         : 25   ← 25
  interval_metrics  : 200   ← R x 100 = 200
  model_selection   : 11   ← 1 + R x B = 11
  point_summary     : 100   ← 100
  method_comparison : 50   ← 50

interval_metrics key 중복 수 : 0   ← 0이어야 함
length = upper - lower 최대 오차 : 1.3322676295501878e-15
point_summary n_evaluated : [2]   ← 모두 R = 2
coverage_cp_minus_mc 최대 오차 : 0.0
config sigma2_hat : 1.0637955954459866  completed_r : [1, 2]


## 12. 검증

계산이 설계대로 되는지 **답을 미리 아는 값**과 비교한다. 가능한 한 위에서 만든 실제 smoke 결과를 다시 계산해 비교한다 (별도 복사본 코드를 시험하면 실제 코드를 검증하지 못하므로).

| 셀 | 확인 내용 |
| --- | --- |
| V1 | CP: 정렬한 score의 k번째 값(보간 없음), k > n_cal이면 inf |
| V2 | MC: 분위수가 B 축(열마다)으로 계산됨 |
| V3 | OOB 선택 규칙: OOB MSE 최소, 동률이면 grid 순서상 먼저 |
| V4 | OOB 예측이 없는 관측치 → 경고가 오류로 바뀜 |
| V5 | 결과표 구조: 평가 Y 공유, CP 길이 공통, 95% ⊇ 90%, 매 r 재생성, 학습 횟수 |
| V6 | 난수 키: 실행 순서와 무관하게 같은 키 → 같은 난수, 단계가 다르면 다른 난수 |

관측 coverage가 정확히 0.90/0.95이거나 CP ≥ MC여야 한다는 검증은 하지 않는다.

In [31]:
# V1. CP 순위 선택 — r = 1 calibration을 같은 키로 다시 만들어 저장된 CP 길이와 비교
random_seed = make_random_state("linear_homo_gaussian", "cp_calibration", r = 1)
check_X = random_seed.uniform(-1, 1, size=(N_CAL, 2))
check_Y = mean_function("linear_homo_gaussian", check_X) + draw_error("linear_homo_gaussian", random_seed, N_CAL)
check_scores = np.sort(np.abs(check_Y - best_model.predict(check_X)))

r1_cp = interval_metrics[(interval_metrics["r"] == 1) & (interval_metrics["method"] == "cp")]
for alpha in ALPHAS:
    saved_q = r1_cp[r1_cp["alpha"] == alpha]["length"].iloc[0] / 2   # 저장된 CP 길이 / 2 = q̂
    k = int(np.ceil((N_CAL + 1) * (1 - alpha)))
    print(f"alpha={alpha}  k={k}")
    print("   저장된 q̂             :", saved_q)
    print(f"   정렬 score {k}번째   :", check_scores[k - 1], "  ← 이것과 같아야 함")
    print(f"   정렬 score {k-1}번째 :", check_scores[k - 2])
    print(f"   정렬 score {k+1}번째 :", check_scores[k])
    print("   np.quantile 보간값   :", np.quantile(check_scores, 1 - alpha), "  ← 보간은 쓰지 않음")

# k > n_cal 경계: n_cal = 10인 작은 예제 (score = 1, 2, ..., 10)
print()
small_scores = np.arange(1, 11)
small_n = len(small_scores)
for alpha in ALPHAS:
    k = int(np.ceil((small_n + 1) * (1 - alpha)))
    if k > small_n:
        q = np.inf
    else:
        q = small_scores[k - 1]
    print(f"n=10, alpha={alpha}: k={k}, q̂={q}")
print("  ← alpha=0.10은 k=10 → q̂=10, alpha=0.05는 k=11 > 10 → q̂=inf 이어야 함")

alpha=0.1  k=901
   저장된 q̂             : 1.6947398511543836
   정렬 score 901번째   : 1.6947398511543839   ← 이것과 같아야 함
   정렬 score 900번째 : 1.6920748709722682
   정렬 score 902번째 : 1.6956666169421155


   np.quantile 보간값   : 1.6923413689904798   ← 보간은 쓰지 않음
alpha=0.05  k=951
   저장된 q̂             : 1.9298992109125441
   정렬 score 951번째   : 1.929899210912544   ← 이것과 같아야 함
   정렬 score 950번째 : 1.9089393155397314
   정렬 score 952번째 : 1.9440170763521394
   np.quantile 보간값   : 1.909987310308371   ← 보간은 쓰지 않음

n=10, alpha=0.1: k=10, q̂=10
n=10, alpha=0.05: k=11, q̂=inf
  ← alpha=0.10은 k=10 → q̂=10, alpha=0.05는 k=11 > 10 → q̂=inf 이어야 함


In [32]:
# V2. MC 분위수 축 — 열(grid)마다 값 범위가 완전히 다른 작은 배열 (행 = b 3개, 열 = grid 4개)
toy_samples = np.array([
    [1.0, 10.0, 100.0, 1000.0],
    [2.0, 20.0, 200.0, 2000.0],
    [3.0, 30.0, 300.0, 3000.0],
])
toy_lower = np.quantile(toy_samples, 0.05, axis=0, method=MC_QUANTILE_METHOD)
toy_upper = np.quantile(toy_samples, 0.95, axis=0, method=MC_QUANTILE_METHOD)
print("toy 하한 :", toy_lower)
print("toy 상한 :", toy_upper)
print("  ← 열마다 [1~3], [10~30], [100~300], [1000~3000] 안에 있어야 함 (다른 열 값이 섞이지 않음)")

# 실제 결과: §10에서 저장한 r = 1 MC 구간이 9-4에서 직접 계산한 mc_lower, mc_upper와 같은지
r1_mc = interval_metrics[(interval_metrics["r"] == 1) & (interval_metrics["method"] == "mc")]
print()
for alpha in ALPHAS:
    saved = r1_mc[r1_mc["alpha"] == alpha].sort_values("grid_id")
    print(f"alpha={alpha}: 하한 일치 {np.allclose(saved['lower'].values, mc_lower[alpha])},"
          f" 상한 일치 {np.allclose(saved['upper'].values, mc_upper[alpha])}")

toy 하한 : [   1.1   11.   110.  1100. ]
toy 상한 : [   2.9   29.   290.  2900. ]
  ← 열마다 [1~3], [10~30], [100~300], [1000~3000] 안에 있어야 함 (다른 열 값이 섞이지 않음)

alpha=0.1: 하한 일치 True, 상한 일치 True
alpha=0.05: 하한 일치 True, 상한 일치 True


In [33]:
# V3. OOB 선택 규칙 — 선택된 조합이 OOB 표에서 OOB MSE가 가장 작은 행인지
# idxmin()은 최솟값이 여러 개면 첫 행을 준다 → "최소, 동률이면 grid 순서상 먼저"와 같은 규칙
# (실제 RF로 정확한 동률을 만들기는 어려워, 동률 처리는 fit_rf_oob의 `<` 비교(같으면 교체 안 함)로 보장된다)
for name, params, table in [("기준 RF", best_params, oob_table), ("MC r=1, b=B", mc_params, mc_table)]:
    best_row = table.loc[table["oob_mse"].idxmin()]
    print(name)
    print("   선택된 조합       :", params)
    print("   OOB MSE 최소 행   :", (int(best_row["min_samples_leaf"]), float(best_row["max_features"]), float(best_row["max_samples"])),
          " oob_mse =", best_row["oob_mse"])
    print("   최소 OOB MSE 개수 :", (table["oob_mse"] == table["oob_mse"].min()).sum(), "  ← 1이면 동률 없음")

기준 RF
   선택된 조합       : (20, 0.5, 1.0)
   OOB MSE 최소 행   : (20, 0.5, 1.0)  oob_mse = 1.044270307881376
   최소 OOB MSE 개수 : 1   ← 1이면 동률 없음
MC r=1, b=B
   선택된 조합       : (20, 0.5, 0.5)
   OOB MSE 최소 행   : (20, 0.5, 0.5)  oob_mse = 1.1606539550209771
   최소 OOB MSE 개수 : 1   ← 1이면 동률 없음


In [34]:
# V4. OOB 예측이 없는 관측치 → 오류로 멈추는지
# 트리 1개 RF: 부트스트랩에 뽑힌 관측치는 OOB 트리가 하나도 없어 sklearn이 경고를 낸다 → §7의 필터로 오류가 되어야 함
tiny_rf = RandomForestRegressor(
    n_estimators=1,
    bootstrap=True,
    oob_score=True,
    random_state=make_random_state("linear_homo_gaussian", "common_training"),
)

raised = False
try:
    tiny_rf.fit(X_train[:50], Y_train[:50])
except UserWarning as error:
    raised = True
    print("오류로 멈춤 :", type(error).__name__, "-", error)

assert raised, "OOB 누락 경고가 오류로 바뀌지 않았음"

오류로 멈춤 : UserWarning - Some inputs do not have OOB scores. This probably means too few trees were used to compute any reliable OOB estimates.


In [35]:
# V5. 결과표 구조 (§10에서 만든 interval_metrics, model_selection 사용)

# ① 같은 (r, grid)의 4행(MC/CP × 두 alpha)이 같은 평가 Y를 쓰는지
n_y_per_point = interval_metrics.groupby(["r", "grid_id"])["y_true"].nunique()
print("① (r, grid)별 y_true 고유값 수 :", n_y_per_point.unique(), "  ← [1]이어야 함")
assert (n_y_per_point == 1).all()

# ② 같은 (r, alpha)에서 CP 길이가 grid 25개 모두 같은지
cp_rows = interval_metrics[interval_metrics["method"] == "cp"]
# 길이 = (f̂+q̂) - (f̂-q̂)라 부동소수점 끝자리(1e-15 수준) 차이는 생길 수 있다 → 최대 - 최소로 본다
cp_length = cp_rows.groupby(["r", "alpha"])["length"]
print("② (r, alpha)별 CP 길이 최대-최소 :", (cp_length.max() - cp_length.min()).max(), "  ← 0 또는 1e-15 수준이어야 함")

# ③ 같은 표본으로 만든 95% 구간이 90% 구간을 포함하는지 (MC, CP 각각)
wide = interval_metrics.pivot_table(index=["r", "grid_id", "method"], columns="alpha", values=["lower", "upper"])
contains = (wide[("lower", 0.05)] <= wide[("lower", 0.10)]) & (wide[("upper", 0.10)] <= wide[("upper", 0.05)])
print("③ 95% ⊇ 90% 인 행 :", contains.groupby("method").sum().to_dict(), f"  ← 방법마다 R x 25 = {R * N_GRID}")

# ④ 매 r마다 평가 Y와 calibration(→ q̂)이 새로 만들어졌는지
y_by_r = interval_metrics[(interval_metrics["method"] == "mc") & (interval_metrics["alpha"] == 0.10)]
y_r1 = y_by_r[y_by_r["r"] == 1]["y_true"].values
y_r2 = y_by_r[y_by_r["r"] == 2]["y_true"].values
print("④ r=1 vs r=2 y_true 같은 grid 수 :", (y_r1 == y_r2).sum(), "/", N_GRID, "  ← 0이어야 함")
print("   r별 CP 반길이 q̂ (alpha=0.10) :", (cp_rows[cp_rows["alpha"] == 0.10].groupby("r")["length"].first() / 2).to_dict())

# ⑤ MC 학습 횟수: 선택 모델 R x B개, 후보 학습 R x B x 24회 (기준 RF·CV fold와 구분)
n_mc_selected = (model_selection["stage"] == "mc").sum()
n_candidates = len(mc_table)   # fit_rf_oob 한 번에 학습하는 후보 수
print("⑤ MC 선택 모델 수 :", n_mc_selected, f"  ← R x B = {R * B}")
print("   후보 조합 수     :", n_candidates, "  ← 24")
print("   MC 후보 학습 수  :", n_mc_selected * n_candidates, f"  ← R x B x 24 = {R * B * 24}")
print("   기준 RF 선택     :", (model_selection["stage"] == "base").sum(), "개 (후보 24회 학습), CV fold RF", CV_FOLDS, "개는 별도")

① (r, grid)별 y_true 고유값 수 : [1]   ← [1]이어야 함
② (r, alpha)별 CP 길이 최대-최소 : 8.881784197001252e-16   ← 0 또는 1e-15 수준이어야 함
③ 95% ⊇ 90% 인 행 : {'cp': 50, 'mc': 50}   ← 방법마다 R x 25 = 50
④ r=1 vs r=2 y_true 같은 grid 수 : 0 / 25   ← 0이어야 함
   r별 CP 반길이 q̂ (alpha=0.10) : {1: 1.6947398511543836, 2: 1.7157856221433914}
⑤ MC 선택 모델 수 : 10   ← R x B = 10
   후보 조합 수     : 24   ← 24
   MC 후보 학습 수  : 240   ← R x B x 24 = 240
   기준 RF 선택     : 1 개 (후보 24회 학습), CV fold RF 10 개는 별도


In [36]:
# V6. 난수 키 — 같은 키는 실행 순서와 무관하게 같은 난수, 키가 다르면 다른 난수

# 같은 키 (mc_dataset, r=2, b=3)를 먼저 뽑고, 다른 키 몇 개를 뽑은 뒤 다시 뽑는다
first = make_random_state("linear_homo_gaussian", "mc_dataset", r = 2, b = 3).normal(size=3)
make_random_state("linear_homo_gaussian", "mc_dataset", r = 1, b = 1).normal(size=1000)
make_random_state("nonlinear_homo_student_t", "test_error", r = 5).normal(size=25)
again = make_random_state("linear_homo_gaussian", "mc_dataset", r = 2, b = 3).normal(size=3)
print("같은 키, 처음  :", first)
print("같은 키, 나중에:", again, "  ← 위와 같아야 함")

# 같은 r, b에서 단계만 다르면 다른 난수 (MC 새 관측오차 vs 실제 test 오차 등)
print()
print("mc_new_error r=1, b=1 :", make_random_state("linear_homo_gaussian", "mc_new_error", r = 1, b = 1).normal(size=3))
print("test_error   r=1      :", make_random_state("linear_homo_gaussian", "test_error", r = 1).normal(size=3))
print("mc_dataset   r=1, b=1 :", make_random_state("linear_homo_gaussian", "mc_dataset", r = 1, b = 1).normal(size=3))
print("  ← 세 줄이 서로 달라야 함")

# scenario만 다르면 다른 난수 (같은 단계·r)
print()
print("linear_homo_gaussian    test_error r=1 :", make_random_state("linear_homo_gaussian", "test_error", r = 1).normal(size=3))
print("linear_homo_student_t   test_error r=1 :", make_random_state("linear_homo_student_t", "test_error", r = 1).normal(size=3))
print("  ← 두 줄이 달라야 함")

같은 키, 처음  : [-0.93924251 -1.19563574 -0.42214473]
같은 키, 나중에: [-0.93924251 -1.19563574 -0.42214473]   ← 위와 같아야 함

mc_new_error r=1, b=1 : [ 0.95609826  0.52227494 -0.35164637]
test_error   r=1      : [0.25869451 0.4046809  0.96237301]
mc_dataset   r=1, b=1 : [-0.41353722  0.18692448 -0.72085204]
  ← 세 줄이 서로 달라야 함

linear_homo_gaussian    test_error r=1 : [0.25869451 0.4046809  0.96237301]
linear_homo_student_t   test_error r=1 : [0.23071049 2.18686181 0.37319343]
  ← 두 줄이 달라야 함
